### Creating project schema.Setting up 'climate_project' as the permanent namespace holding all bronze/silver/gold tables.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS climate_project")

DataFrame[]

### Declaring all 8 raw columns upfront instead of letting Spark infer types.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("ID", StringType(), True),
    StructField("DATE", StringType(), True),
    StructField("ELEMENT", StringType(), True),
    StructField("DATA_VALUE", StringType(), True),
    StructField("M_FLAG", StringType(), True),
    StructField("Q_FLAG", StringType(), True),
    StructField("S_FLAG", StringType(), True),
    StructField("OBS_TIME", StringType(), True),
])

### Pulling 2016–2026 from S3 using the explicit schema. 2016–2025 = baseline period.

In [0]:
years = list(range(2016, 2027))
paths = [f"s3://noaa-ghcn-pds/csv/by_year/{y}.csv" for y in years]

df_bronze = spark.read.csv(paths, header=True, schema=schema)

print(df_bronze.count())

391479581


### Saving the raw, untouched data permanently as a Delta table

In [0]:
df_bronze.write.format("delta").mode("overwrite").saveAsTable("climate_project.bronze_observations")

### Peeking the table to see evething is as it is. 

In [0]:
df_bronze_read = spark.table("climate_project.bronze_observations")
print(df_bronze_read.count())

391479581
